# Radionica 1 — Uvod u podatke i vizualizaciju
### GirlTHing ML/Data Science blok

Danas radimo sa datasetom "stvarnih" oglasa za prodaju stanova u Tuzli. Cilj nije da napamet znate pandas naredbe — cilj je da naučite **proces** koji možete ponoviti na bilo kom dataset-u:

**struktura → kvalitet podataka → jedna varijabla → dvije varijable → grupe/segmenti → zaključak**

Radite u paru. Kad naiđete na `# TODO`, to je mjesto gdje vi dovršavate kod — sve ostalo je već pripremljeno da radi.

## Korak 0 — Učitavanje podataka

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

URL = "https://drive.google.com/uc?export=download&id=152Ff9y6aQsctxMRNEZqiiy9wsFH6Tkdu"
df = pd.read_csv(URL)



In [ ]:
# TODO: prikaži prvih 5 redova
df.___()

## Korak 1 — Struktura dataset-a
Prije bilo čega drugog: koliko redova/kolona imamo, koji su tipovi podataka, i koliko toga uopšte nedostaje?

In [ ]:
# TODO: ispiši oblik dataset-a (broj redova, broj kolona)
print("Broj redova i kolona:", ___)

# TODO: ispiši info o tipovima podataka i broju ne-null vrijednosti po koloni
df.___()

In [ ]:
# TODO: deskriptivna statistika za sve kolone (i brojevne i tekstualne)
df.describe(include=___)

## Korak 2 — Kvalitet podataka (čišćenje)

Ovaj dataset ima nekoliko tipičnih problema pravih oglasa. Za svaki problem koji riješite, **imenujte ga naglas paru** — nedostajuća vrijednost? Duplikat? Pogrešan tip? Outlier?

### 2a — Koliko nedostaje po koloni?

In [ ]:
# TODO: prebroji nedostajuće vrijednosti po koloni
df.___().sum()

### 2b — Cijena je upisana kao tekst u više formata
Pogledajte kolonu `cijena_km` — neki redovi su brojevi, neki tekst poput `"85.000 KM"` ili `"85 000"`. Napravite funkciju koja sve te formate pretvara u čist broj (float).

In [ ]:
def parse_cijena(v):
    if pd.isna(v):
        return np.nan
    s = str(v).upper().replace('KM', '').strip().replace(' ', '')
    # Pažnja: '.' i ',' mogu biti separator hiljada (128.100 = 128100) ILI
    # obična decimala kad broj već dolazi kao npr. 98900.0 (tj. .0 na kraju).
    # TODO: ako string završava na '.0', to je decimala - ukloni samo taj sufiks.
    # Inače, ukloni SVE tačke i zareze (to su separatori hiljada).
    if s.endswith(___):
        s = s[:-2]
    else:
        s = s.replace(___, '').replace(___, '')
    try:
        return float(s)
    except ValueError:
        return np.nan

# TODO: primijeni funkciju na kolonu 'cijena_km' i sačuvaj u novu kolonu 'cijena_km_clean'
df['cijena_km_clean'] = df['cijena_km'].apply(___)
df[['cijena_km', 'cijena_km_clean']].sample(8)

### 2c — Kvadratura kao tekst, nekonzistentni nazivi naselja
Slična priča: `kvadratura_m2` ponegdje ima `"45 m2"` umjesto broja, a `naselje` ima velika/mala slova, razmake i dodatke poput `"- Tuzla"`.

In [ ]:
def parse_kvadratura(v):
    if pd.isna(v):
        return np.nan
    s = str(v).lower().replace('m2', '').replace('m²', '').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan

def clean_naselje(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    # TODO: ukloni dodatak '- Tuzla' ako postoji
    s = s.replace(___, '')
    # TODO: ujednači veliko/malo slovo (hint: .title())
    return s.strip().___()

df['kvadratura_clean'] = df['kvadratura_m2'].apply(___)
df['naselje_clean'] = df['naselje'].apply(___)
df['naselje_clean'].value_counts()

### 2d — Sprat kao tekst ("Prizemlje") i godina kao raspon ("1970-1979")
Dva dodatna, vrlo autentična problema iz stvarnih OLX oglasa: prizemlje se ponekad piše riječju umjesto brojem 0, a godina izgradnje ponekad dolazi kao raspon decenije umjesto tačne godine.

In [ ]:
import re

def parse_sprat(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    # TODO: ako je tekst 'prizemlje', to je sprat 0
    if s == ___:
        return 0
    try:
        return float(s)
    except ValueError:
        return np.nan

def parse_godina(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    # TODO: ako string sadrži '-' ili 'do', to je raspon (npr. '1970-1979') - uzmi sredinu kao procjenu
    if '-' in s or 'do' in s:
        nums = [int(x) for x in re.findall(r'\d{4}', s)]
        return float(np.mean(___)) if nums else np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

# TODO: primijeni obje funkcije
df['sprat_clean'] = df['sprat'].apply(___)
df['godina_clean'] = df['godina_izgradnje'].apply(___)
df[['sprat', 'sprat_clean', 'godina_izgradnje', 'godina_clean']].sample(8)

### 2e — Kolona `stanje` zapravo miješa dva različita pojma — ovdje VI pišete funkciju (uz pomoć AI-ja)

Pogledajte pažljivije moguće vrijednosti u koloni `stanje`:

`Useljivo, Renoviran, Potrebna renovacija, Dobro stanje, Nov, Namješten, Nenamješten`

Primjećujete problem? Ova kolona miješa **fizičko stanje stana** (nov, renoviran, treba renovaciju...) i **namještenost** (namješten/nenamješten) — dvije different stvari nabačene u jednu kolonu. Ovo je vrlo česta situacija u stvarnim podacima.

**Za razliku od prethodnih koraka, ovdje nema praznina za popuniti — napišite funkcije od nule, uz pomoć AI alata (Copilot, ChatGPT, šta god koristite):**

1. Funkcija `parse_fizicko_stanje(v)` — iz `stanje` izdvaja SAMO fizičko stanje (`Nov`, `Renoviran`, `Potrebna renovacija`, `Useljivo`, `Dobro stanje`); za `Namješten`/`Nenamješten` (i za NaN) vraća `NaN`, jer ti redovi ne govore ništa o fizičkom stanju.
2. Funkcija `parse_namjestenost(v)` — izdvaja SAMO namještenost (`Namješten`/`Nenamješten`); za sve ostalo vraća `NaN`.
3. Primijenite obje na kolonu `stanje`, sačuvajte kao nove kolone `fizicko_stanje` i `namjestenost_clean`.
4. Provjerite: koliko NaN vrijednosti ima svaka nova kolona? Da li vam se to čini logično?

**Kako iskoristiti AI:** opišite mu problem svojim riječima ("imam pandas kolonu koja ..."), pogledajte šta predloži, pa **provjerite** da li stvarno radi na par primjera prije nego nastavite dalje — AI ponekad pogriješi (npr. zaboravi NaN, ili pogrešno kategoriše neku vrijednost).

In [ ]:
# Podsjetnik na moguće vrijednosti u 'stanje':
print(df['stanje'].value_counts(dropna=False))

# Probaj sama, uz pomoć AI-ja:
# def parse_fizicko_stanje(v):
#     ...
#
# def parse_namjestenost(v):
#     ...

# df['fizicko_stanje'] = df['stanje'].apply(...)
# df['namjestenost_clean'] = df['stanje'].apply(...)

# Provjera:
# print(df['fizicko_stanje'].isna().sum())
# print(df['namjestenost_clean'].isna().sum())


### 2f — Duplikati i outlieri
Neki stanovi su objavljeni dvaput (isti detalji, drugi ID oglasa). Neki redovi imaju očigledne greške u unosu (npr. cijena od 1 KM, stan od 4 m², sprat 45).

In [ ]:
# TODO: pronađi duplirane redove (isti stan, različit oglas_id) - koristi kolone koje NISU oglas_id
dupli = df.duplicated(subset=[___], keep=False)
print("Broj redova uključenih u duplikate:", dupli.sum())

# TODO: filtriraj razumne opsege (cijena/m2 između 1000 i 10000 KM, kvadratura između 15 i 250, sprat <= 20)
df_clean = df[___].copy()

# TODO: ukloni duplikate
df_clean = df_clean.drop_duplicates(subset=[___])

df_clean['cijena_m2'] = df_clean['cijena_km_clean'] / df_clean['kvadratura_clean']
print("Redova prije čišćenja:", len(df), "| poslije:", len(df_clean))

## Korak 3 — Prvi grafovi

**Podsjetnik — koji graf za koje pitanje:**
- Jedna brojevna varijabla → **histogram**
- Brojevna varijabla po kategorijama → **boxplot**
- Odnos dvije brojevne varijable → **scatter**
- Puno korelacija odjednom → **heatmap**

### 3a — Distribucija cijene po m²

In [ ]:
plt.figure(figsize=(8, 4))
# TODO: nacrtaj histogram kolone 'cijena_m2' iz df_clean, 30 binova
sns.___(df_clean[___], bins=___)
plt.title('Cijena po m² — distribucija')
plt.xlabel('KM/m²')
plt.show()

### 3b — Cijena/m² po fizičkom stanju stana (boxplot)

Koristimo `fizicko_stanje` (kolonu koju ste upravo napravili) umjesto sirove `stanje` kolone — tako namještenost ne miješa kategorije na grafu.

In [ ]:
plt.figure(figsize=(9, 5))
# TODO: napravi boxplot - x='fizicko_stanje', y='cijena_m2', data=df_clean
sns.boxplot(data=___, x=___, y=___)
plt.xticks(rotation=20)
plt.title('Cijena/m² prema stanju stana')
plt.show()

> **Usput — korelacija nije uzročnost.** Prije nego zaključite da nešto "uzrokuje" nešto drugo iz grafa, pitajte se: postoji li neki treći faktor koji objašnjava oboje?

## Korak 4 — Filtriranje i sortiranje
### 4a — 10 najskupljih stanova (po ukupnoj cijeni)

In [ ]:
# TODO: sortiraj df_clean po 'cijena_km_clean' opadajuće, prikaži prvih 10
df_clean.sort_values(___, ascending=___)[
    ['naselje_clean', 'kvadratura_clean', 'sprat_clean', 'stanje', 'cijena_km_clean']
].head(___)

### 4b — Prosječna cijena/m² po naselju (groupby)

In [ ]:
# TODO: grupiši po 'naselje_clean', izračunaj prosjek 'cijena_m2', sortiraj opadajuće
df_clean.groupby(___)[___].mean().sort_values(ascending=___).round(0)

### Checkpoint 🎯
Prođite kratko sa parom: **šta vas je već sada iznenadilo u podacima?** Recite jednu rečenicu naglas.

## Izazov — Šta određuje cijenu stana?

**Zadatak:** pronađite jedan iznenađujući uvid o tome šta utiče na cijenu stana, i pripremite jedan graf koji to dokazuje + 1-2 rečenice objašnjenja (bez koda — obično jezik).

Ako ste zapele, probajte jedno od ovih pitanja:
- Da li veći stanovi imaju nižu ili višu cijenu po m² od manjih?
- Da li stanje stana (renovirano vs treba renovaciju) mijenja cijenu više nego lokacija?
- Da li sprat utiče na cijenu (prizemlje/najviši sprat vs srednji)?
- Ima li veze između godine izgradnje i cijene?

**Kako predstaviti graf (3 pravila):**
1. Naslov = poenta, ne opis ("Noviji stanovi su skuplji", ne "Cijena vs godina")
2. Ukloni sve što ne nosi informaciju (nepotrebne linije, boje, 3D efekte)
3. Istakni jednu stvar — ne pokušavaj reći sve na jednom grafu

In [ ]:
# TODO: vaš graf ovdje (scatter, boxplot ili heatmap - birajte prema pitanju koje istražujete)
plt.figure(figsize=(9, 6))
___
plt.title(___)  # naslov = poenta!
plt.show()

In [ ]:
# Prostor za dodatno istraživanje ako imate vremena
___

## Prezentacija 🎤
60-90 sekundi po paru: pokažite graf, recite uvid naglas.

---
## Zadaća
**Dio 1:** Instalirajte VS Code + Jupyter ekstenziju, aktivirajte GitHub Copilot, i pokušajte proširiti današnju EDA vježbu uz njegovu pomoć. Zapišite: gdje vam je pomogao, gdje je predložio nešto pogrešno.

**Dio 2:** Pronađite (ili napravite) svoj dataset. Ponovite proces sa časa: učitajte, riješite bar 2 problema u podacima, napravite 2-3 grafa, napišite jedan pasus sa najzanimljivijim uvidom.

Vidimo se na Radionici 2 — hajmo vidjeti da li model može predvidjeti cijenu bolje od vas! 🚀